# Data Preparation

Prepare the encounter data for analysis. This section applies basic cleaning and audits the resulting CSV.

## Setup and Raw Data Loading

Load the verified source encounters from the local raw CSV. The loader validates the input file, schema, and original labels.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

project_root = next(
    directory
    for directory in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preparation.data_cleaning import clean_encounters, save_cleaning_result
from src.preparation.data_loading import load_raw_encounters

encounters = load_raw_encounters()

## Basic Data Cleaning

Apply the configured marker replacements. Each configured marker becomes an empty CSV field. The laboratory category `None` retains its literal representation.

In [ ]:
cleaned = clean_encounters(encounters)
changed_cells = sum(row["changed_cells"] for row in cleaned.audit)
print(f"Encounters: {len(encounters.rows):,} before, {len(cleaned.rows):,} after")
print(f"Columns: {len(encounters.columns):,} before, {len(cleaned.columns):,} after")
print(f"Changed cells: {changed_cells:,}")

audit_table = [
    "| Column | Source marker | Markers before | Empty fields before | Empty fields after | Literal retained |",
    "| --- | --- | ---: | ---: | ---: | --- |",
]
for row in cleaned.audit:
    if row["source_marker"] or row["preserved_literal"]:
        marker = f"`{row['source_marker']}`" if row["source_marker"] else ""
        literal = f"`{row['preserved_literal']}` ({row['literal_after']:,})" if row["preserved_literal"] else ""
        audit_table.append(
            f"| `{row['column']}` | {marker} | {row['marker_before']:,} | "
            f"{row['empty_before']:,} | {row['empty_after']:,} | {literal} |"
        )
display(Markdown("\n".join(audit_table)))

In [ ]:
cleaned_path, audit_path = save_cleaning_result(cleaned)
print(cleaned_path.relative_to(project_root))
print(audit_path.relative_to(project_root))

## Cohort Selection

Exclude the configured death and hospice discharge codes before selecting one eligible encounter per patient. The smallest numeric encounter ID identifies the representative record. The source lacks encounter timestamps for ordering records by date.

In [ ]:
from src.preparation.data_loading import load_ids_mapping
from src.preparation.dataset_preparation import save_cohort_result, select_cohort

cohort = select_cohort(cleaned.columns, cleaned.rows, load_ids_mapping())

cohort_table = [
    "| Stage | Encounters | Patients | Removed at this step | `NO` | `<30` | `>30` |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for row in cohort.flow:
    cohort_table.append(
        f"| {row['stage']} | {row['encounters']:,} | {row['patients']:,} | "
        f"{row['removed']:,} | {row['NO']:,} | {row['<30']:,} | {row['>30']:,} |"
    )
display(Markdown("\n".join(cohort_table)))

for heading, counts in (
    ("Excluded encounters by discharge code", cohort.excluded_by_code),
    ("Cohort encounters with unclear discharge descriptions", cohort.unclear_retained_by_code),
):
    print(heading)
    for code, count in counts.items():
        print(f"  {code}: {count:,}")

In [ ]:
missing_table = [
    "| Column | Cleaned empty fields | Eligible empty fields | Cohort empty fields |",
    "| --- | ---: | ---: | ---: |",
]
for row in cohort.audit:
    if row["cleaned_empty"]:
        missing_table.append(
            f"| `{row['column']}` | {row['cleaned_empty']:,} | "
            f"{row['eligible_empty']:,} | {row['cohort_empty']:,} |"
        )
display(Markdown("\n".join(missing_table)))

cohort_path, cohort_audit_path = save_cohort_result(cohort)
print(cohort_path.relative_to(project_root))
print(cohort_audit_path.relative_to(project_root))

## Diagnosis Grouping

Apply the fixed ICD-9 research grouping to the three diagnosis fields. The grouped fields take the place of the source diagnosis fields in the transformed table. Empty source values map to `Missing`, and values that fail parsing map to `Unmapped`.

In [ ]:
from src.preparation.dataset_preparation import DIAGNOSIS_GROUPS, group_diagnoses

grouped_cohort = group_diagnoses(cohort.columns, cohort.rows)
print(f"Encounters: {len(cohort.rows):,} before, {len(grouped_cohort.rows):,} after")
print(f"Columns: {len(cohort.columns):,} before, {len(grouped_cohort.columns):,} after")

group_columns = list(grouped_cohort.group_counts)
group_table = [
    "| Diagnosis group | " + " | ".join(f"`{column}`" for column in group_columns) + " |",
    "| --- |" + " ---: |" * len(group_columns),
]
for group in DIAGNOSIS_GROUPS:
    counts = [f"{grouped_cohort.group_counts[column][group]:,}" for column in group_columns]
    group_table.append(f"| `{group}` | " + " | ".join(counts) + " |")
display(Markdown("\n".join(group_table)))